# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.datePublished}")
print(f"License: {meta.license}")
print(f"Keywords: {', '.join(meta.keywords) if hasattr(meta, 'keywords') else ''}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll list all record sets and their IDs, then for each record set, enumerate its fields and their `@id` and `name` if available.

In [ ]:
# Collect information about record sets and fields
record_sets = dataset.metadata.recordSet
if record_sets is None or len(record_sets) == 0:
    # Single-record-set fallback for older Croissant versions
    print("No recordSets defined in top-level metadata. Attempting to find via .distribution...")
    record_sets = []
    if hasattr(dataset.metadata, 'distribution') and dataset.metadata.distribution:
        # Sometimes record set is referenced in distribution[0]["encoding"]
        recs = getattr(dataset.metadata.distribution[0], 'recordSet', None)
        if recs:
            record_sets = recs if isinstance(recs, list) else [recs]
else:
    # If found in metadata
    record_sets = record_sets if isinstance(record_sets, list) else [record_sets]

if not record_sets:
    # Use helper method to extract record set IDs from metadata JSON
    rec_ids = []
    metadata_json = dataset.metadata.to_json() if hasattr(dataset.metadata, 'to_json') else dataset.metadata
    if 'recordSet' in metadata_json:
        if isinstance(metadata_json['recordSet'], list):
            for rs in metadata_json['recordSet']:
                rec_ids.append(rs['@id'])
        elif isinstance(metadata_json['recordSet'], dict):
            rec_ids.append(metadata_json['recordSet']['@id'])
    record_sets = rec_ids

if not record_sets:
    raise ValueError('No record sets found via metadata. Cannot continue.')

print("Available record sets and fields:")
overview = {}
for rec_set in record_sets:
    # rec_set can be an id string or an object with @id
    rec_id = rec_set['@id'] if isinstance(rec_set, dict) and '@id' in rec_set else rec_set
    print(f'---\nRecordSet @id: {rec_id}')
    fields = dataset.fields(record_set=rec_id)
    field_ids = []
    for f in fields:
        print(f"   Field @id: {f['@id']}, name: {f.get('name')}, type: {f.get('dataType')}")
        field_ids.append(f['@id'])
    overview[rec_id] = field_ids

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll demonstrate loading one record set (the first one listed) and display its available columns.

In [ ]:
# Pick the first record set for demonstration
main_record_set_id = list(overview.keys())[0]
print(f"Using record set: {main_record_set_id}")

# Load all records from this record set into a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
# If records present, convert to DataFrame
df = pd.DataFrame(records)
print(f"Columns in {main_record_set_id}:")
print(df.columns.tolist())
print("\nPreview:")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field for filtering and normalization. Replace `<numeric_field_id>` and `<group_field_id>` with appropriate field `@id`s from your actual record set/fields.

In [ ]:
# Select a numeric field for analysis (edit as appropriate for your dataset)
numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
if not numeric_fields:
    print("No numeric columns found in dataframe.")
else:
    numeric_field = numeric_fields[0]  # e.g. select molecular weight or similar
    print(f"Using numeric field: {numeric_field}")

    threshold = df[numeric_field].mean() if df[numeric_field].notna().any() else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try using a string or categorical field to group
    cat_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
    if len(cat_fields) > 0:
        group_field = cat_fields[0]  # e.g. a protein name/identifier
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical grouping column found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example histogram and boxplot for one numeric field.

In [ ]:
if not numeric_fields:
    print("No numeric fields to visualize.")
else:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')

    plt.subplot(1,2,2)
    df.boxplot(column=numeric_field)
    plt.title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and previewed data from the FAIR² "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using `mlcroissant`.
- Identified available record sets and their fields using `@id` referencing for robust processing.
- Demonstrated basic EDA including selection and normalization of numeric fields and grouping by categorical attributes.
- Provided visualizations for initial data distribution assessment.

For further analysis, consider more in-depth domain-specific filtering, additional visualizations, and downstream modeling using this dataset.